STEP 0. 라이브러리 및 데이터 불러오기

In [3]:

# ============================================================
# STEP 0. 데이터 불러오기 및 공통 설정
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import genpareto, kstest
from IPython.display import display
from pathlib import Path

DATA_DIR = Path("C:/Users/alpin/Desktop/승혁/baf/26-2")

# 공통 임계값
U1_Q = 0.87

U2_PROBS = {
    "95": 0.95,
    "97.5": 0.975,
    "98": 0.98,
    "99": 0.99,
    "99.5": 0.995
}

MAIN_MODEL = "KRW_7"
MODEL_ORDER = ["USD_4", "KRW_4", "USD_7", "KRW_7"]

MODEL_LABELS = {
    "USD_4": "원본 KCCI / 위기 4개",
    "KRW_4": "환율 적용 / 위기 4개",
    "USD_7": "원본 KCCI / 위기 7개",
    "KRW_7": "환율 적용 / 위기 7개"
}

ROUTE_LABELS = {
    "KAUI": "호주",
    "KCI": "중국",
    "KJI": "일본",
    "KLEI": "중남미 동안",
    "KLWI": "중남미 서안",
    "KMEI": "중동",
    "KSAI": "남아공",
    "KSEI": "동남아",
    "KUEI": "미국 동안",
    "KUWI": "미국 서안",
    "KWAI": "서아프리카",
    "REDSEA_WEIGHTED": "홍해 통합"
}

ROUTE_WEIGHTS = {
    "KUWI": 0.15,
    "KUEI": 0.10,
    "REDSEA_WEIGHTED": 0.15,
    "KMEI": 0.05,
    "KAUI": 0.05,
    "KLEI": 0.05,
    "KLWI": 0.05,
    "KSAI": 0.025,
    "KWAI": 0.025,
    "KCI": 0.15,
    "KJI": 0.10,
    "KSEI": 0.10
}

assert np.isclose(sum(ROUTE_WEIGHTS.values()), 1.0)

# 데이터 불러오기
df_normal4 = pd.read_excel(
    DATA_DIR / "KCCI_정상주4개.xlsx",
    sheet_name="데이터"
)

df_test4 = pd.read_excel(
    DATA_DIR / "KCCI_test주4개.xlsx",
    sheet_name="데이터"
)

df_normal7 = pd.read_excel(
    DATA_DIR / "KCCI_정상주7개.xlsx",
    sheet_name="데이터"
)

df_test7 = pd.read_excel(
    DATA_DIR / "KCCI_test주7개.xlsx",
    sheet_name="데이터"
)

# 날짜 및 금액 형식 통일
for df in [df_normal4, df_test4, df_normal7, df_test7]:
    for col in ["KCCI_DATE", "EVENT_START", "END_EXCLUSIVE"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    for col in ["KCCI_USD_PER_FEU", "KCCI_KRW_PER_FEU"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 모형별 데이터
CONFIG = {
    "USD_4": (df_normal4, df_test4, "KCCI_USD_PER_FEU"),
    "KRW_4": (df_normal4, df_test4, "KCCI_KRW_PER_FEU"),
    "USD_7": (df_normal7, df_test7, "KCCI_USD_PER_FEU"),
    "KRW_7": (df_normal7, df_test7, "KCCI_KRW_PER_FEU")
}

# 데이터 검증
data_check = pd.DataFrame([
    {"구분": "정상주 / 위기 4개 제외", "관측치": len(df_normal4)},
    {"구분": "TEST / 위기 4개", "관측치": len(df_test4)},
    {"구분": "정상주 / 위기 7개 제외", "관측치": len(df_normal7)},
    {"구분": "TEST / 위기 7개", "관측치": len(df_test7)}
])

assert len(df_normal4) + len(df_test4) == 2340
assert len(df_normal7) + len(df_test7) == 2340

for model, (normal, test, col) in CONFIG.items():
    assert normal[col].notna().all(), f"{model}: 정상주 운임 결측"
    assert test[col].notna().all(), f"{model}: TEST 운임 결측"

print("■ 데이터 구성 및 검증")
display(data_check)

print(f"U1 분위수: {U1_Q:.0%}")
print("최종 U2 기준: 98%")
print("최종 상품 설계 후보: 환율 적용 KCCI / 위기 7개")


■ 데이터 구성 및 검증


,구분,관측치
0,정상주 / 위기 4개 제외,2269
1,TEST / 위기 4개,71
2,정상주 / 위기 7개 제외,2224
3,TEST / 위기 7개,116


U1 분위수: 87%
최종 U2 기준: 98%
최종 상품 설계 후보: 환율 적용 KCCI / 위기 7개


STEP 1. 정상주 12주 Baseline 및 상승률 계산
 U1 : 0.87 기준

In [4]:

# ============================================================
# STEP 1. 정상주 Baseline 초기화 및 상승률 계산
# ============================================================

models = {}
full_data = {}

for model, (normal, test, value_col) in CONFIG.items():

    # 전체 시계열 복원
    full = pd.concat(
        [
            normal[["KCCI_DATE", "ROUTE", value_col]],
            test[["KCCI_DATE", "ROUTE", value_col]]
        ],
        ignore_index=True
    )

    full = full.sort_values(
        ["ROUTE", "KCCI_DATE"]
    ).reset_index(drop=True)

    if full.duplicated(["ROUTE", "KCCI_DATE"]).any():
        raise ValueError(f"{model}: 항로·날짜 중복")

    full["OBS_NO"] = full.groupby("ROUTE").cumcount()

    # 실제 TEST용 이동 Baseline
    # 위기주도 포함된 전체 시계열에서 계산
    full["ROLL_BASE"] = (
        full.groupby("ROUTE")[value_col]
        .transform(
            lambda s: s.shift(1).rolling(
                12, min_periods=12
            ).mean()
        )
    )

    full["ROLL_R"] = (
        full[value_col] / full["ROLL_BASE"] - 1
    )

    full_data[model] = full

    # 정상주만 분리
    train = normal[
        ["KCCI_DATE", "ROUTE"]
    ].merge(
        full[[
            "KCCI_DATE", "ROUTE", "OBS_NO", value_col
        ]],
        on=["KCCI_DATE", "ROUTE"],
        how="left",
        validate="one_to_one"
    )

    train = train.rename(
        columns={value_col: "KCCI"}
    )

    train = train.sort_values(
        ["ROUTE", "KCCI_DATE"]
    ).reset_index(drop=True)

    # 위기기간 때문에 관측 순서가 끊기면 초기화
    gap = (
        train.groupby("ROUTE")["OBS_NO"]
        .diff().ne(1)
    )

    train["SEGMENT"] = (
        gap.groupby(train["ROUTE"]).cumsum()
    )

    # 정상구간별 직전 12개 관측치 평균
    train["BASELINE"] = (
        train.groupby(["ROUTE", "SEGMENT"])["KCCI"]
        .transform(
            lambda s: s.shift(1).rolling(
                12, min_periods=12
            ).mean()
        )
    )

    train["R"] = (
        train["KCCI"] / train["BASELINE"] - 1
    )

    models[model] = train

# 요약표
baseline_check = pd.DataFrame([
    {
        "분석 모형": MODEL_LABELS[model],
        "정상주 관측치": len(data),
        "상승률 계산 가능 주":
            int(data["R"].notna().sum()),
        "정상 구간 수":
            data.groupby(["ROUTE", "SEGMENT"]).ngroups
    }
    for model, data in models.items()
])

print("■ 정상주 Baseline 계산 결과")
display(baseline_check)


■ 정상주 Baseline 계산 결과


,분석 모형,정상주 관측치,상승률 계산 가능 주,정상 구간 수
0,원본 KCCI / 위기 4개,2269,2082,16
1,환율 적용 / 위기 4개,2269,2082,16
2,원본 KCCI / 위기 7개,2224,2002,19
3,환율 적용 / 위기 7개,2224,2002,19


STEP 2. 항로별 U1 87% 분위수 계산

In [5]:

# ============================================================
# STEP 2. 항로별 U1 = 87% 분위수
# ============================================================

u1_results = {}

for model, data in models.items():

    u1 = (
        data.groupby("ROUTE")["R"]
        .agg(
            N="count",
            U1=lambda x: x.quantile(U1_Q)
        )
        .reset_index()
    )

    u1_results[model] = u1

# 최종 후보 KRW_7 확인
u1_view = u1_results[MAIN_MODEL].copy()

u1_view["U1"] *= 100
u1_view["ROUTE"] = u1_view["ROUTE"].replace(
    ROUTE_LABELS
)

u1_view = u1_view.rename(columns={
    "ROUTE": "항로",
    "N": "정상주 상승률 관측치",
    "U1": "U1 기준(%)"
})

print("■ 환율 적용 KCCI / 위기 7개 / U1 87%")
display(u1_view.round(3))


■ 환율 적용 KCCI / 위기 7개 / U1 87%


,항로,정상주 상승률 관측치,U1 기준(%)
0,호주,183,36.339
1,중국,183,10.060
2,일본,183,6.205
3,중남미 동안,183,51.873
4,중남미 서안,183,45.316
5,중동,153,13.857
6,남아공,183,16.544
7,동남아,183,21.092
8,미국 동안,78,32.800
9,미국 서안,160,40.384


STEP 3. U1 초과값 추출

In [6]:

# ============================================================
# STEP 3. 정상주 U1 초과값
# ============================================================

exceedance_results = {}

for model, data in models.items():

    temp = data.merge(
        u1_results[model][["ROUTE", "U1"]],
        on="ROUTE",
        how="left",
        validate="many_to_one"
    )

    temp["EXCESS"] = (
        temp["R"] - temp["U1"]
    ).where(temp["R"] > temp["U1"])

    exceedance_results[model] = temp

sample_check = pd.DataFrame([
    {
        "분석 모형": MODEL_LABELS[model],
        "U1 분위수": "87%",
        "U1 초과 표본 수":
            int(data["EXCESS"].notna().sum())
    }
    for model, data in exceedance_results.items()
])

print("■ U1 초과값 추출 결과")
display(sample_check)


■ U1 초과값 추출 결과


,분석 모형,U1 분위수,U1 초과 표본 수
0,원본 KCCI / 위기 4개,87%,273
1,환율 적용 / 위기 4개,87%,273
2,원본 KCCI / 위기 7개,87%,263
3,환율 적용 / 위기 7개,87%,263


STEP 4. GPD 적합 및 모수 추정
위기 주 제외한 정상주 데이터(train data)로 U1값 추출

In [7]:

# ============================================================
# STEP 4. GPD 적합
# ============================================================

gpd_results = {}
gpd_rows = []

for model, data in exceedance_results.items():

    excess = data["EXCESS"].dropna().to_numpy(
        dtype=float
    )

    excess = excess[
        np.isfinite(excess) & (excess > 0)
    ]

    if len(excess) < 30:
        raise ValueError(
            f"{model}: 초과 표본 수가 부족합니다."
        )

    # 위치 모수 0으로 고정
    xi, _, sigma = genpareto.fit(
        excess, floc=0
    )

    # 일반 KS 검정 (참고용)
    ks_stat, ks_p = kstest(
        excess,
        "genpareto",
        args=(xi, 0, sigma)
    )

    gpd_results[model] = {
        "xi": xi,
        "sigma": sigma,
        "excess": excess,
        "ks_stat": ks_stat,
        "ks_p": ks_p
    }

    gpd_rows.append({
        "MODEL": model,
        "N_EXCESS": len(excess),
        "SHAPE_XI": xi,
        "SCALE_SIGMA": sigma,
        "KS_STAT": ks_stat,
        "KS_P_VALUE": ks_p
    })

gpd_summary = pd.DataFrame(gpd_rows)

gpd_view = gpd_summary.rename(columns={
    "MODEL": "분석 모형",
    "N_EXCESS": "초과 표본 수",
    "SHAPE_XI": "Shape ξ",
    "SCALE_SIGMA": "Scale σ",
    "KS_STAT": "KS 통계량",
    "KS_P_VALUE": "일반 KS p값"
})

gpd_view["분석 모형"] = gpd_view["분석 모형"].replace(
    MODEL_LABELS
)

print("■ U1 87% 기준 GPD 적합 결과")
display(gpd_view.round(5))


■ U1 87% 기준 GPD 적합 결과


,분석 모형,초과 표본 수,Shape ξ,Scale σ,KS 통계량,일반 KS p값
0,원본 KCCI / 위기 4개,273,0.28673,0.14459,0.05004,0.48573
1,환율 적용 / 위기 4개,273,0.23026,0.15904,0.03770,0.81844
2,원본 KCCI / 위기 7개,263,0.31525,0.13739,0.04506,0.64276
3,환율 적용 / 위기 7개,263,0.26333,0.15017,0.03943,0.79302


STEP 5. Bootstrap KS 적합도 검정

In [8]:

# ============================================================
# STEP 5. Parametric Bootstrap KS
# ============================================================

def bootstrap_ks(excess, B=1000, seed=42):

    rng = np.random.default_rng(seed)

    xi, _, sigma = genpareto.fit(excess, floc=0)

    observed_ks = kstest(
        excess,
        "genpareto",
        args=(xi, 0, sigma)
    ).statistic

    simulated_ks = np.empty(B)

    for i in range(B):

        sample = genpareto.rvs(
            c=xi,
            loc=0,
            scale=sigma,
            size=len(excess),
            random_state=rng
        )

        xi_b, _, sigma_b = genpareto.fit(
            sample, floc=0
        )

        simulated_ks[i] = kstest(
            sample,
            "genpareto",
            args=(xi_b, 0, sigma_b)
        ).statistic

    p_boot = (
        1 + np.count_nonzero(
            simulated_ks >= observed_ks
        )
    ) / (B + 1)

    return observed_ks, p_boot


for model in MODEL_ORDER:

    ks_stat, p_boot = bootstrap_ks(
        gpd_results[model]["excess"],
        B=1000,
        seed=42
    )

    gpd_results[model]["ks_p_boot"] = p_boot

    gpd_summary.loc[
        gpd_summary["MODEL"] == model,
        "KS_P_BOOT"
    ] = p_boot

# 한글 결과
gpd_ks_view = gpd_summary.rename(columns={
    "MODEL": "분석 모형",
    "N_EXCESS": "초과 표본 수",
    "SHAPE_XI": "Shape ξ",
    "SCALE_SIGMA": "Scale σ",
    "KS_STAT": "KS 통계량",
    "KS_P_VALUE": "일반 KS p값",
    "KS_P_BOOT": "Bootstrap KS p값"
})

gpd_ks_view["분석 모형"] = (
    gpd_ks_view["분석 모형"].replace(MODEL_LABELS)
)

gpd_ks_view["5% 검정 판단"] = np.where(
    gpd_ks_view["Bootstrap KS p값"] < 0.05,
    "GPD 적합 가설 기각",
    "기각하지 못함"
)

print("■ Bootstrap KS 적합도 검정")
display(gpd_ks_view.round(5))


■ Bootstrap KS 적합도 검정


,분석 모형,초과 표본 수,Shape ξ,Scale σ,KS 통계량,일반 KS p값,Bootstrap KS p값,5% 검정 판단
0,원본 KCCI / 위기 4개,273,0.28673,0.14459,0.05004,0.48573,0.08691,기각하지 못함
1,환율 적용 / 위기 4개,273,0.23026,0.15904,0.03770,0.81844,0.48651,기각하지 못함
2,원본 KCCI / 위기 7개,263,0.31525,0.13739,0.04506,0.64276,0.21079,기각하지 못함
3,환율 적용 / 위기 7개,263,0.26333,0.15017,0.03943,0.79302,0.43157,기각하지 못함


STEP 6. GPD 기반 U2 전체 재계산

In [9]:

# ============================================================
# STEP 6. GPD 기반 U2 재계산
# ============================================================

u2_probability_results = {}
gpd_quantile_rows = []

for model in MODEL_ORDER:

    xi = gpd_results[model]["xi"]
    sigma = gpd_results[model]["sigma"]

    result = u1_results[model][["ROUTE", "U1"]].copy()
    extra_row = {"MODEL": model}

    for label, p in U2_PROBS.items():

        assert p > U1_Q

        # 전체 분포 분위수 → 조건부 GPD 분위수
        tail_q = (p - U1_Q) / (1 - U1_Q)

        extra = genpareto.ppf(
            tail_q,
            c=xi,
            loc=0,
            scale=sigma
        )

        result[f"U2_{label}"] = result["U1"] + extra
        extra_row[f"EXCESS_{label}"] = extra

    u2_probability_results[model] = result
    gpd_quantile_rows.append(extra_row)

gpd_quantiles = pd.DataFrame(gpd_quantile_rows)

# 한글 출력
view = gpd_quantiles[["MODEL", "EXCESS_98"]].copy()

view["MODEL"] = view["MODEL"].replace(MODEL_LABELS)
view["EXCESS_98"] *= 100

view = view.rename(columns={
    "MODEL": "분석 모형",
    "EXCESS_98": "U2 98% GPD 초과폭(%p)"
})

print("■ U1 87% 기준 U2 재계산 결과")
display(view.round(3))

print(
    "U2 98%의 GPD 조건부 분위수:",
    f"{(0.98-U1_Q)/(1-U1_Q):.4%}"
)


■ U1 87% 기준 U2 재계산 결과


,분석 모형,U2 98% GPD 초과폭(%p)
0,원본 KCCI / 위기 4개,35.822
1,환율 적용 / 위기 4개,37.215
2,원본 KCCI / 위기 7개,35.045
3,환율 적용 / 위기 7개,36.330


U2 98%의 GPD 조건부 분위수: 84.6154%


STEP 7. 항로별 U1·U2 기준 비교표

In [10]:

# ============================================================
# STEP 7. 항로별 U1·U2 비교
# ============================================================

for model in ["USD_7", "KRW_7"]:

    view = u2_probability_results[model].copy()

    cols = ["U1"] + [
        f"U2_{p}" for p in U2_PROBS
    ]

    view[cols] = view[cols] * 100

    view["ROUTE"] = view["ROUTE"].replace(
        ROUTE_LABELS
    )

    view = view.rename(columns={
        "ROUTE": "항로",
        "U1": "U1 기준(%)",
        "U2_95": "U2 95%(%)",
        "U2_97.5": "U2 97.5%(%)",
        "U2_98": "U2 98%(%)",
        "U2_99": "U2 99%(%)",
        "U2_99.5": "U2 99.5%(%)"
    })

    print(f"\n■ {MODEL_LABELS[model]} 임계값")
    display(view.reset_index(drop=True).round(2))



■ 원본 KCCI / 위기 7개 임계값


,항로,U1 기준(%),U2 95%(%),U2 97.5%(%),U2 98%(%),U2 99%(%),U2 99.5%(%)
0,호주,36.54,51.86,66.25,71.59,90.79,114.68
1,중국,8.81,24.13,38.52,43.86,63.06,86.95
2,일본,5.56,20.88,35.27,40.61,59.81,83.70
3,중남미 동안,51.86,67.18,81.57,86.91,106.11,130.00
4,중남미 서안,43.92,59.24,73.63,78.97,98.17,122.06
5,중동,15.01,30.33,44.72,50.06,69.26,93.15
6,남아공,16.61,31.93,46.31,51.65,70.86,94.75
7,동남아,20.17,35.48,49.87,55.21,74.41,98.31
8,미국 동안,37.08,52.40,66.79,72.13,91.33,115.22
9,미국 서안,39.23,54.54,68.93,74.27,93.47,117.37



■ 환율 적용 / 위기 7개 임계값


,항로,U1 기준(%),U2 95%(%),U2 97.5%(%),U2 98%(%),U2 99%(%),U2 99.5%(%)
0,호주,36.34,52.65,67.34,72.67,91.36,113.80
1,중국,10.06,26.38,41.06,46.39,65.09,87.52
2,일본,6.20,22.52,37.21,42.54,61.23,83.67
3,중남미 동안,51.87,68.19,82.88,88.20,106.90,129.34
4,중남미 서안,45.32,61.63,76.32,81.65,100.34,122.78
5,중동,13.86,30.17,44.86,50.19,68.88,91.32
6,남아공,16.54,32.86,47.55,52.87,71.57,94.01
7,동남아,21.09,37.41,52.09,57.42,76.12,98.56
8,미국 동안,32.80,49.12,63.80,69.13,87.83,110.26
9,미국 서안,40.38,56.70,71.39,76.71,95.41,117.85


STEP 8. 정상주 U2 초과 여부 확인
정상기간에 각 U2 수준을 초과한 주가 얼마나 되는지 계산

In [11]:

# ============================================================
# STEP 8. 정상기간 U2 초과
# ============================================================

normal_eval = {}

for model in MODEL_ORDER:

    data = models[model].dropna(
        subset=["R"]
    ).copy()

    thresholds = u2_probability_results[model][
        ["ROUTE"] + [
            f"U2_{p}" for p in U2_PROBS
        ]
    ]

    data = data.merge(
        thresholds,
        on="ROUTE",
        how="left",
        validate="many_to_one"
    )

    data = data.sort_values(
        ["ROUTE", "SEGMENT", "KCCI_DATE"]
    )

    for p in U2_PROBS:

        data[f"HIT_{p}"] = (
            data["R"] >= data[f"U2_{p}"]
        )

    normal_eval[model] = data

# 최종 후보 KRW_7
rows = []

for p in U2_PROBS:

    data = normal_eval[MAIN_MODEL]

    rows.append({
        "U2 기준": f"{p}%",
        "정상주 상승률 관측치": len(data),
        "U2 초과 관측 주 수":
            int(data[f"HIT_{p}"].sum()),
        "정상주 초과율(%)":
            100 * data[f"HIT_{p}"].mean()
    })

print("■ KRW_7 정상기간 U2 초과 빈도")
display(pd.DataFrame(rows).round(3))


■ KRW_7 정상기간 U2 초과 빈도


,U2 기준,정상주 상승률 관측치,U2 초과 관측 주 수,정상주 초과율(%)
0,95%,2002,105,5.245
1,97.5%,2002,57,2.847
2,98%,2002,46,2.298
3,99%,2002,23,1.149
4,99.5%,2002,10,0.500


STEP 9. 정상기간 독립 초과 구간 및 항로 가중빈도
U2를 3주 연속 넘었다면 초과 관측 주 수=3, 독립 초과 구간 수=1로 계산(하나의 단위로 계산)

In [12]:

# ============================================================
# STEP 9. 정상기간 독립 U2 초과 구간
# ============================================================

normal_rows = []

for model in MODEL_ORDER:

    data = normal_eval[model].copy()

    for p in U2_PROBS:

        temp = data[[
            "ROUTE", "SEGMENT", "KCCI_DATE",
            "R", f"HIT_{p}"
        ]].copy()

        temp["HIT"] = temp[f"HIT_{p}"].astype(bool)

        # 직전 정상주 발동 여부
        prev_hit = (
            temp.groupby(["ROUTE", "SEGMENT"])["HIT"]
            .shift(1, fill_value=False)
        )

        # 연속 발동은 최초 한 번만 카운트
        temp["NEW_EPISODE"] = (
            temp["HIT"] & ~prev_hit
        )

        route_stats = (
            temp.groupby("ROUTE", as_index=False)
            .agg(
                N=("R", "size"),
                HIT_WEEKS=("HIT", "sum"),
                EPISODES=("NEW_EPISODE", "sum")
            )
        )

        route_stats["WEIGHT"] = (
            route_stats["ROUTE"].map(
                ROUTE_WEIGHTS
            )
        )

        if route_stats["WEIGHT"].isna().any():
            raise ValueError("항로 가중치 누락")

        weighted_hit = (
            52 * route_stats["WEIGHT"]
            * route_stats["HIT_WEEKS"]
            / route_stats["N"]
        ).sum()

        weighted_episode = (
            52 * route_stats["WEIGHT"]
            * route_stats["EPISODES"]
            / route_stats["N"]
        ).sum()

        normal_rows.append({
            "MODEL": model,
            "U2_PROB": f"{p}%",
            "NORMAL_N": int(route_stats["N"].sum()),
            "TOTAL_HIT_WEEKS":
                int(route_stats["HIT_WEEKS"].sum()),
            "TOTAL_EPISODES":
                int(route_stats["EPISODES"].sum()),
            "WEIGHTED_ANNUAL_HIT_WEEKS":
                float(weighted_hit),
            "WEIGHTED_ANNUAL_EPISODES":
                float(weighted_episode)
        })

normal_episode_summary = pd.DataFrame(normal_rows)

view = normal_episode_summary[
    normal_episode_summary["MODEL"] == MAIN_MODEL
].copy()

view = view.rename(columns={
    "MODEL": "분석 모형",
    "U2_PROB": "U2 기준",
    "NORMAL_N": "정상주 상승률 관측치",
    "TOTAL_HIT_WEEKS": "U2 초과 주 수",
    "TOTAL_EPISODES": "독립 초과 구간 수",
    "WEIGHTED_ANNUAL_HIT_WEEKS":
        "가중 연평균 초과 주 수",
    "WEIGHTED_ANNUAL_EPISODES":
        "가중 연평균 독립 초과 구간 수"
})

view["분석 모형"] = view["분석 모형"].replace(
    MODEL_LABELS
)

print("■ KRW_7 정상주 U2 초과 빈도")
display(view.reset_index(drop=True).round(3))


■ KRW_7 정상주 U2 초과 빈도


,분석 모형,U2 기준,정상주 상승률 관측치,U2 초과 주 수,독립 초과 구간 수,가중 연평균 초과 주 수,가중 연평균 독립 초과 구간 수
0,환율 적용 / 위기 7개,95%,2002,105,22,2.607,0.576
1,환율 적용 / 위기 7개,97.5%,2002,57,14,1.395,0.401
2,환율 적용 / 위기 7개,98%,2002,46,13,1.036,0.400
3,환율 적용 / 위기 7개,99%,2002,23,6,0.374,0.120
4,환율 적용 / 위기 7개,99.5%,2002,10,3,0.092,0.043


STEP 10. GPD의 U2 추정 오차 진단

In [13]:

# ============================================================
# STEP 10. GPD 추정 U2 vs 경험적 분위수
# ============================================================

tail_q = (0.98 - U1_Q) / (1 - U1_Q)

rows = []

for model in ["USD_7", "KRW_7"]:

    x = np.sort(
        gpd_results[model]["excess"]
    )

    xi = gpd_results[model]["xi"]
    sigma = gpd_results[model]["sigma"]

    # 실제 초과값 분위수
    empirical = np.quantile(x, tail_q)

    # GPD 추정 분위수
    predicted = genpareto.ppf(
        tail_q,
        c=xi,
        loc=0,
        scale=sigma
    )

    n = len(x)

    fitted = genpareto.cdf(
        x, c=xi, loc=0, scale=sigma
    )

    d_plus = np.arange(1, n+1)/n - fitted
    d_minus = fitted - np.arange(n)/n

    if d_plus.max() >= d_minus.max():
        idx = int(np.argmax(d_plus))
        empirical_cdf = (idx + 1) / n
    else:
        idx = int(np.argmax(d_minus))
        empirical_cdf = idx / n

    rows.append({
        "분석 모형": MODEL_LABELS[model],
        "초과 표본 수": n,
        "경험적 분위수 초과폭(%p)":
            empirical * 100,
        "GPD 추정 초과폭(%p)":
            predicted * 100,
        "추정 차이(%p)":
            (predicted - empirical) * 100,
        "최대 KS 발생 초과폭(%p)":
            x[idx] * 100,
        "해당 지점 경험적 누적확률(%)":
            empirical_cdf * 100,
        "해당 지점 GPD 누적확률(%)":
            fitted[idx] * 100
    })

print("■ U2 98% 부근 GPD 추정 오차 진단")
display(pd.DataFrame(rows).round(4))


■ U2 98% 부근 GPD 추정 오차 진단


,분석 모형,초과 표본 수,경험적 분위수 초과폭(%p),GPD 추정 초과폭(%p),추정 차이(%p),최대 KS 발생 초과폭(%p),해당 지점 경험적 누적확률(%),해당 지점 GPD 누적확률(%)
0,원본 KCCI / 위기 7개,263,38.0977,35.0449,-3.0528,6.1654,38.7833,34.2773
1,환율 적용 / 위기 7개,263,38.7343,36.3305,-2.4038,7.7225,42.2053,38.2624


STEP 11. 실제 위기 사건 정의
위기 7개 조합 각각에 대해 설정된 사건기간과 관측 주 수를 확인

In [14]:

# ============================================================
# STEP 11. 실제 위기 사건 목록
# ============================================================

EVENTS = {}

for scenario, test in [
    ("4", df_test4),
    ("7", df_test7)
]:

    event_list = (
        test.groupby(
            [
                "EVENT_NAME",
                "ROUTE",
                "EVENT_START",
                "END_EXCLUSIVE"
            ],
            dropna=False
        )
        .size()
        .reset_index(name="WINDOW_N")
    )

    if event_list[
        ["EVENT_START", "END_EXCLUSIVE"]
    ].isna().any().any():
        raise ValueError(
            f"위기 {scenario}개 데이터의 날짜 누락"
        )

    EVENTS[scenario] = event_list

view = EVENTS["7"].rename(columns={
    "EVENT_NAME": "위기 사건",
    "ROUTE": "항로",
    "EVENT_START": "사건 시작일",
    "END_EXCLUSIVE": "사건 종료 경계일(미포함)",
    "WINDOW_N": "실제 관측 주 수"
}).copy()

view["항로"] = view["항로"].replace(ROUTE_LABELS)

print("■ TEST 위기·항로 7개 조합")
display(view)


■ TEST 위기·항로 7개 조합


,위기 사건,항로,사건 시작일,사건 종료 경계일(미포함),실제 관측 주 수
0,미국 관세 충격,미국 동안,2025-04-02,2025-06-25,11
1,미국 관세 충격,미국 서안,2025-04-02,2025-06-25,11
2,미국 동부·걸프 항만파업,미국 동안,2024-10-01,2024-12-24,12
3,볼티모어 교량 붕괴,미국 동안,2024-03-26,2024-06-18,12
4,파나마운하 가뭄·통항제한,미국 동안,2023-07-30,2024-01-14,23
5,호르무즈 위기,중동,2026-02-28,2026-08-15,23
6,홍해·수에즈 위기,홍해 통합,2023-12-15,2024-05-31,24


STEP 12. U1 최초 발동 후 12주 U2 백테스트

In [15]:

# ============================================================
# STEP 12. 실제 위기 U1·U2 백테스트
# U1 발동 → Baseline 고정 → 12주 추적
# ============================================================

event_rows = []
followup_rows = []

for model in MODEL_ORDER:

    value_col = CONFIG[model][2]
    scenario = model.split("_")[-1]

    full = full_data[model].copy()

    u1_map = (
        u1_results[model]
        .set_index("ROUTE")["U1"]
    )

    u2_map = (
        u2_probability_results[model]
        .set_index("ROUTE")
    )

    for ev in EVENTS[scenario].itertuples(index=False):

        event = ev.EVENT_NAME
        route = ev.ROUTE
        start = pd.Timestamp(ev.EVENT_START)
        end = pd.Timestamp(ev.END_EXCLUSIVE)

        route_data = full[
            full["ROUTE"] == route
        ].sort_values("KCCI_DATE")

        # U1을 찾는 범위: 설정된 위기기간 안
        event_period = route_data[
            (route_data["KCCI_DATE"] >= start) &
            (route_data["KCCI_DATE"] < end)
        ]

        u1 = float(u1_map.loc[route])

        u1_hits = event_period[
            event_period["ROLL_R"] >= u1
        ]

        u1_on = not u1_hits.empty

        u1_date = pd.NaT
        frozen_base = np.nan

        trace = route_data.iloc[0:0].copy()

        if u1_on:

            first = u1_hits.iloc[0]
            u1_date = first["KCCI_DATE"]

            frozen_base = float(
                first["ROLL_BASE"]
            )

            # U1 최초 발동일 + 12주
            followup_end = (
                u1_date + pd.Timedelta(weeks=12)
            )

            # 사건 종료와 관계없이 전체 시계열에서 추출
            trace = route_data[
                (route_data["KCCI_DATE"] >= u1_date) &
                (route_data["KCCI_DATE"] < followup_end)
            ].copy()

            trace["FROZEN_R"] = (
                trace[value_col] / frozen_base - 1
            )

            trace["MODEL"] = model
            trace["EVENT"] = event
            trace["FROZEN_BASE"] = frozen_base
            trace["U1_DATE"] = u1_date

            followup_rows.append(trace)

        # 주간 KCCI의 관측일이 며칠씩 변동할 수 있으므로
        # 마지막 관측일이 시작 후 최소 11주 경과했는지 확인
        followup_complete = bool(
            len(trace) >= 12
            and trace["KCCI_DATE"].max()
            >= u1_date + pd.Timedelta(weeks=11)
        ) if u1_on else False

        # 5개 U2 기준별로 평가
        for p in U2_PROBS:

            u2 = float(
                u2_map.loc[route, f"U2_{p}"]
            )

            if u1_on:
                u2_hits = trace[
                    trace["FROZEN_R"] >= u2
                ]
            else:
                u2_hits = trace

            u2_on = not u2_hits.empty

            u2_date = (
                u2_hits.iloc[0]["KCCI_DATE"]
                if u2_on else pd.NaT
            )

            event_rows.append({
                "MODEL": model,
                "EVENT": event,
                "ROUTE": route,
                "EVENT_START": start,
                "EVENT_END_EXCLUSIVE": end,
                "U2_PROB": f"{p}%",
                "U1_THRESHOLD": u1,
                "U2_THRESHOLD": u2,
                "U1_TRIGGER": int(u1_on),
                "U1_DATE": u1_date,
                "FROZEN_BASE": frozen_base,
                "U2_TRIGGER": int(u2_on),
                "U2_DATE": u2_date,
                "U2_DELAY_WEEKS": (
                    (u2_date - start).days / 7
                    if u2_on else np.nan
                ),
                "U2_AFTER_U1_WEEKS": (
                    (u2_date - u1_date).days / 7
                    if u2_on else np.nan
                ),
                "FOLLOWUP_OBS": len(trace),
                "FOLLOWUP_COMPLETE": followup_complete,
                "U2_AFTER_EVENT_END": bool(
                    u2_on and u2_date >= end
                )
            })

event_detail = pd.DataFrame(event_rows)

event_followup_weekly = (
    pd.concat(followup_rows, ignore_index=True)
    if followup_rows else pd.DataFrame()
)

# KRW_7, U2 98% 결과만 출력
view = event_detail[
    (event_detail["MODEL"] == MAIN_MODEL) &
    (event_detail["U2_PROB"] == "98%")
].copy()

view = view[[
    "EVENT", "ROUTE",
    "U1_DATE", "U2_DATE",
    "U2_AFTER_U1_WEEKS",
    "U2_TRIGGER",
    "FOLLOWUP_COMPLETE",
    "U2_AFTER_EVENT_END"
]]

view = view.rename(columns={
    "EVENT": "위기 사건",
    "ROUTE": "항로",
    "U1_DATE": "U1 최초 발동일",
    "U2_DATE": "U2 최초 발동일",
    "U2_AFTER_U1_WEEKS": "U1 이후 소요 주 수",
    "U2_TRIGGER": "U2 탐지",
    "FOLLOWUP_COMPLETE": "12주 추적 완료",
    "U2_AFTER_EVENT_END": "사건기간 종료 후 U2 발동"
})

view["항로"] = view["항로"].replace(ROUTE_LABELS)

for col in [
    "U2 탐지",
    "12주 추적 완료",
    "사건기간 종료 후 U2 발동"
]:
    view[col] = view[col].replace({
        1: "O", 0: "X",
        True: "O", False: "X"
    })

print("■ KRW_7 / U1 87% / U2 98% 사건별 결과")
display(view.reset_index(drop=True).round(2))


■ KRW_7 / U1 87% / U2 98% 사건별 결과


,위기 사건,항로,U1 최초 발동일,U2 최초 발동일,U1 이후 소요 주 수,U2 탐지,12주 추적 완료,사건기간 종료 후 U2 발동
0,미국 관세 충격,미국 동안,2025-06-09,NaT,NaN,X,True,False
1,미국 관세 충격,미국 서안,2025-06-09,2025-06-09,0.0,O,True,False
2,미국 동부·걸프 항만파업,미국 동안,NaT,NaT,NaN,X,False,False
3,볼티모어 교량 붕괴,미국 동안,2024-06-03,2024-07-01,4.0,O,True,True
4,파나마운하 가뭄·통항제한,미국 동안,2024-01-08,2024-01-15,1.0,O,True,True
5,호르무즈 위기,중동,2026-03-09,2026-03-09,0.0,O,False,False
6,홍해·수에즈 위기,홍해 통합,2023-12-26,2024-01-02,1.0,O,True,False


STEP 13. 정상주와 실제 위기 종합 성능 비교

In [16]:

# ============================================================
# STEP 13. 정상기간 + 실제 위기 종합 비교
# ============================================================

event_summary = (
    event_detail.groupby(
        ["MODEL", "U2_PROB"],
        as_index=False
    )
    .agg(
        N_EVENTS=("U2_TRIGGER", "size"),
        U1_DETECTED=("U1_TRIGGER", "sum"),
        U2_DETECTED=("U2_TRIGGER", "sum"),
        MEAN_DELAY_WEEKS=(
            "U2_DELAY_WEEKS", "mean"
        ),
        N_COMPLETE=("FOLLOWUP_COMPLETE", "sum"),
        N_AFTER_END=("U2_AFTER_EVENT_END", "sum")
    )
)

event_summary["U2_DETECTION_RATE_%"] = (
    100 * event_summary["U2_DETECTED"]
    / event_summary["N_EVENTS"]
)

comparison = normal_episode_summary.merge(
    event_summary,
    on=["MODEL", "U2_PROB"],
    how="inner",
    validate="one_to_one"
)

comparison_kr = comparison.rename(columns={
    "MODEL": "분석 모형",
    "U2_PROB": "U2 기준",
    "NORMAL_N": "정상주 관측치",
    "TOTAL_HIT_WEEKS": "정상주 U2 초과 주 수",
    "TOTAL_EPISODES": "정상주 독립 초과 구간",
    "WEIGHTED_ANNUAL_HIT_WEEKS":
        "가중 연평균 초과 주 수",
    "WEIGHTED_ANNUAL_EPISODES":
        "가중 연평균 독립 초과 구간",
    "N_EVENTS": "전체 위기 조합 수",
    "U1_DETECTED": "U1 탐지 건수",
    "U2_DETECTED": "U2 탐지 건수",
    "MEAN_DELAY_WEEKS": "평균 탐지기간(주)",
    "N_COMPLETE": "12주 추적 완료 건수",
    "N_AFTER_END": "사건기간 종료 후 U2 도달 건수",
    "U2_DETECTION_RATE_%": "U2 탐지율(%)"
})

comparison_kr["분석 모형"] = (
    comparison_kr["분석 모형"].replace(MODEL_LABELS)
)

print("■ 최종 후보 KRW_7 / U2별 성능 비교")

display(
    comparison_kr[
        comparison_kr["분석 모형"]
        == MODEL_LABELS[MAIN_MODEL]
    ].reset_index(drop=True).round(3)
)

print("■ U2 98% / 4개 모형 비교")

display(
    comparison_kr[
        comparison_kr["U2 기준"] == "98%"
    ].reset_index(drop=True).round(3)
)


■ 최종 후보 KRW_7 / U2별 성능 비교


,분석 모형,U2 기준,정상주 관측치,정상주 U2 초과 주 수,정상주 독립 초과 구간,가중 연평균 초과 주 수,가중 연평균 독립 초과 구간,전체 위기 조합 수,U1 탐지 건수,U2 탐지 건수,평균 탐지기간(주),12주 추적 완료 건수,사건기간 종료 후 U2 도달 건수,U2 탐지율(%)
0,환율 적용 / 위기 7개,95%,2002,105,22,2.607,0.576,7,6,6,9.714,5,1,85.714
1,환율 적용 / 위기 7개,97.5%,2002,57,14,1.395,0.401,7,6,6,10.048,5,2,85.714
2,환율 적용 / 위기 7개,98%,2002,46,13,1.036,0.400,7,6,5,10.314,5,2,71.429
3,환율 적용 / 위기 7개,99%,2002,23,6,0.374,0.120,7,6,4,11.214,5,2,57.143
4,환율 적용 / 위기 7개,99.5%,2002,10,3,0.092,0.043,7,6,3,10.286,5,1,42.857


■ U2 98% / 4개 모형 비교


,분석 모형,U2 기준,정상주 관측치,정상주 U2 초과 주 수,정상주 독립 초과 구간,가중 연평균 초과 주 수,가중 연평균 독립 초과 구간,전체 위기 조합 수,U1 탐지 건수,U2 탐지 건수,평균 탐지기간(주),12주 추적 완료 건수,사건기간 종료 후 U2 도달 건수,U2 탐지율(%)
0,원본 KCCI / 위기 4개,98%,2082,46,12,0.987,0.322,4,3,3,6.238,2,1,75.000
1,환율 적용 / 위기 4개,98%,2082,47,13,1.025,0.360,4,3,3,6.238,2,1,75.000
2,원본 KCCI / 위기 7개,98%,2002,43,11,0.938,0.319,7,6,6,10.548,5,2,85.714
3,환율 적용 / 위기 7개,98%,2002,46,13,1.036,0.400,7,6,5,10.314,5,2,71.429


STEP 13-1. 위기별 U2 탐지 O/X 비교

In [17]:

# ============================================================
# STEP 13-1. 실제 사건별 U2 탐지표
# ============================================================

P_LABELS = [f"{p}%" for p in U2_PROBS]

for model in ["USD_7", "KRW_7"]:

    temp = event_detail[
        event_detail["MODEL"] == model
    ].copy()

    temp["위기명"] = (
        temp["EVENT"].astype(str)
        + " ("
        + temp["ROUTE"].replace(ROUTE_LABELS)
        + ")"
    )

    pivot = temp.pivot_table(
        index="위기명",
        columns="U2_PROB",
        values="U2_TRIGGER",
        aggfunc="max"
    )

    pivot = pivot.reindex(columns=P_LABELS)

    if pivot.isna().any().any():
        raise ValueError("U2 기준 열 이름 불일치")

    pivot = pivot.replace({1: "O", 0: "X"})
    pivot.columns.name = "U2 기준"

    print(f"\n■ {MODEL_LABELS[model]} 위기별 탐지")
    display(pivot)



■ 원본 KCCI / 위기 7개 위기별 탐지


U2 기준,95%,97.5%,98%,99%,99.5%
위기명,,,,,
미국 관세 충격 (미국 동안),O,O,O,X,X
미국 관세 충격 (미국 서안),O,O,O,O,X
미국 동부·걸프 항만파업 (미국 동안),X,X,X,X,X
볼티모어 교량 붕괴 (미국 동안),O,O,O,O,X
파나마운하 가뭄·통항제한 (미국 동안),O,O,O,O,O
호르무즈 위기 (중동),O,O,O,O,O
홍해·수에즈 위기 (홍해 통합),O,O,O,O,O



■ 환율 적용 / 위기 7개 위기별 탐지


U2 기준,95%,97.5%,98%,99%,99.5%
위기명,,,,,
미국 관세 충격 (미국 동안),O,O,X,X,X
미국 관세 충격 (미국 서안),O,O,O,X,X
미국 동부·걸프 항만파업 (미국 동안),X,X,X,X,X
볼티모어 교량 붕괴 (미국 동안),O,O,O,O,X
파나마운하 가뭄·통항제한 (미국 동안),O,O,O,O,O
호르무즈 위기 (중동),O,O,O,O,O
홍해·수에즈 위기 (홍해 통합),O,O,O,O,O


STEP 14. Monte Carlo Placebo 검정

In [18]:

# ============================================================
# STEP 14. Monte Carlo Placebo 검정
# U1 87% / U2 5개 기준 / 50,000회
# ============================================================

N_MC = 50_000
MC_SEED = 20261008

rng = np.random.default_rng(MC_SEED)

# ------------------------------------------------------------
# 1. 정상주 연속구간 구성
# ------------------------------------------------------------

normal_blocks = {}

for model in MODEL_ORDER:

    for route, route_df in models[model].groupby("ROUTE"):

        blocks = []

        for _, part in route_df.groupby("SEGMENT"):

            part = part.sort_values("KCCI_DATE")

            blocks.append({
                "dates": part["KCCI_DATE"].to_numpy(
                    dtype="datetime64[ns]"
                ),
                "kcci": part["KCCI"].to_numpy(
                    dtype=float
                ),
                "base": part["BASELINE"].to_numpy(
                    dtype=float
                ),
                "r": part["R"].to_numpy(
                    dtype=float
                )
            })

        normal_blocks[(model, route)] = blocks

# ------------------------------------------------------------
# 2. 모형별 U1·U2
# ------------------------------------------------------------

mc_u1 = {
    model: (
        u1_results[model]
        .set_index("ROUTE")["U1"]
        .to_dict()
    )
    for model in MODEL_ORDER
}

mc_u2 = {
    model: (
        u2_probability_results[model]
        .set_index("ROUTE")
    )
    for model in MODEL_ORDER
}

# ------------------------------------------------------------
# 3. 가상 위기별 최대 상승률 계산
# ------------------------------------------------------------

pool_cache = {}

def get_placebo_pool(model, route, window_n):

    key = (model, route, int(window_n))

    if key in pool_cache:
        return pool_cache[key]

    window_n = int(window_n)
    u1 = float(mc_u1[model][route])

    maxima = []

    for block in normal_blocks[(model, route)]:

        dates = block["dates"]
        kcci = block["kcci"]
        base = block["base"]
        r = block["r"]

        # 시작 전 12개 관측치가 있어야 함
        for left in range(
            12, len(r) - window_n + 1
        ):

            right = left + window_n

            # 가상 사건 마지막 주에 U1이 발생해도
            # 그 이후 12주를 관찰할 수 있어야 함
            last_possible_u1 = right - 1

            if dates[-1] < (
                dates[last_possible_u1]
                + np.timedelta64(77, "D")
            ):
                continue

            # 가상 사건기간 내 최초 U1
            hits = np.flatnonzero(
                np.isfinite(r[left:right]) &
                (r[left:right] >= u1)
            )

            if hits.size == 0:
                maxima.append(-np.inf)
                continue

            first = left + int(hits[0])
            frozen_base = base[first]

            if (
                not np.isfinite(frozen_base)
                or frozen_base <= 0
            ):
                raise ValueError(
                    f"{model}/{route}: Baseline 오류"
                )

            # U1 이후 12주간 추적
            followup_end = (
                dates[first] + np.timedelta64(84, "D")
            )

            active = (
                (dates >= dates[first]) &
                (dates < followup_end)
            )

            frozen_r = (
                kcci[active] / frozen_base - 1
            )

            maxima.append(
                float(np.max(frozen_r))
            )

    pool = np.asarray(maxima, dtype=float)

    if pool.size == 0:
        raise ValueError(
            f"{model}/{route}/{window_n}주: "
            "사용 가능한 정상주 Placebo 구간 없음"
        )

    pool_cache[key] = pool

    return pool


# ------------------------------------------------------------
# 4. 실제 위기 탐지 건수
# ------------------------------------------------------------

actual_counts = (
    event_detail.groupby(
        ["MODEL", "U2_PROB"]
    )["U2_TRIGGER"]
    .sum()
)

all_routes = np.array(list(ROUTE_WEIGHTS))

route_probs = np.array([
    ROUTE_WEIGHTS[r] for r in all_routes
])

# ------------------------------------------------------------
# 5. Monte Carlo 실행
# ------------------------------------------------------------

mc_summary_rows = []
mc_frequency_rows = []
placebo_distributions = {}

for mode in [
    "route_matched",
    "all_route_weighted"
]:

    for model in MODEL_ORDER:

        scenario = model.split("_")[-1]
        events = EVENTS[scenario]
        n_events = len(events)

        simulated = np.zeros(
            (N_MC, len(U2_PROBS)),
            dtype=np.int16
        )

        for ev in events.itertuples(index=False):

            window_n = int(ev.WINDOW_N)

            # 방법 A: 동일 항로·기간
            if mode == "route_matched":

                route = ev.ROUTE

                pool = get_placebo_pool(
                    model, route, window_n
                )

                draws = pool[
                    rng.integers(
                        0, len(pool), size=N_MC
                    )
                ]

                u2_values = (
                    mc_u2[model]
                    .loc[
                        route,
                        [f"U2_{p}" for p in U2_PROBS]
                    ]
                    .to_numpy(dtype=float)
                )

                simulated += (
                    draws[:, None] >= u2_values[None, :]
                )

            # 방법 B: 전체 항로 가중
            else:

                sampled_routes = rng.choice(
                    all_routes,
                    size=N_MC,
                    p=route_probs
                )

                for route in all_routes:

                    mask = sampled_routes == route
                    n_draw = int(mask.sum())

                    if n_draw == 0:
                        continue

                    pool = get_placebo_pool(
                        model, route, window_n
                    )

                    draws = pool[
                        rng.integers(
                            0, len(pool), size=n_draw
                        )
                    ]

                    u2_values = (
                        mc_u2[model]
                        .loc[
                            route,
                            [f"U2_{p}" for p in U2_PROBS]
                        ]
                        .to_numpy(dtype=float)
                    )

                    simulated[mask] += (
                        draws[:, None] >= u2_values[None, :]
                    )

        placebo_distributions[(mode, model)] = simulated

        # ----------------------------------------------------
        # 6. U2 분위수별 요약
        # ----------------------------------------------------

        for j, p in enumerate(U2_PROBS):

            label = f"{p}%"

            observed = int(
                actual_counts.loc[(model, label)]
            )

            null = simulated[:, j]

            p_mc = (
                np.count_nonzero(null >= observed) + 1
            ) / (N_MC + 1)

            ci_low, ci_high = np.quantile(
                null, [0.025, 0.975]
            )

            mc_summary_rows.append({
                "MODE": mode,
                "MODEL": model,
                "U2_PROB": label,
                "N_EVENTS": n_events,
                "ACTUAL_DETECTED": observed,
                "ACTUAL_RATE_%":
                    100 * observed / n_events,
                "PLACEBO_MEAN": float(null.mean()),
                "PLACEBO_95_LOW": ci_low,
                "PLACEBO_95_HIGH": ci_high,
                "MC_P_VALUE": p_mc
            })

            counts = np.bincount(
                null,
                minlength=n_events + 1
            )

            for k in range(n_events + 1):

                mc_frequency_rows.append({
                    "MODE": mode,
                    "MODEL": model,
                    "U2_PROB": label,
                    "DETECTED_EVENTS": k,
                    "SIMULATION_COUNT": int(counts[k]),
                    "PROBABILITY_PCT":
                        100 * counts[k] / N_MC
                })

# ------------------------------------------------------------
# 7. 결과 저장 및 검사
# ------------------------------------------------------------

placebo_summary = pd.DataFrame(
    mc_summary_rows
)

placebo_frequency = pd.DataFrame(
    mc_frequency_rows
)

assert len(placebo_summary) == 40

assert (
    placebo_frequency.groupby(
        ["MODE", "MODEL", "U2_PROB"]
    )["SIMULATION_COUNT"].sum() == N_MC
).all()

print("■ Monte Carlo Placebo 계산 완료")
print("반복 횟수: 50,000회")


■ Monte Carlo Placebo 계산 완료
반복 횟수: 50,000회


STEP 15. Monte Carlo 결과 및 최종 모형 성능표
STEP 15-1. Placebo 검정 결과 한글 출력

In [19]:

# ============================================================
# STEP 15-1. Placebo 결과 한글 표
# ============================================================

MODE_LABELS = {
    "route_matched": "동일 항로·기간",
    "all_route_weighted": "전체 KCCI 항로 가중"
}

placebo_summary_kr = placebo_summary.rename(columns={
    "MODE": "플라세보 방식",
    "MODEL": "분석 모형",
    "U2_PROB": "U2 기준",
    "N_EVENTS": "실제 위기 조합 수",
    "ACTUAL_DETECTED": "실제 탐지 건수",
    "ACTUAL_RATE_%": "실제 탐지율(%)",
    "PLACEBO_MEAN": "가상 평균 탐지 건수",
    "PLACEBO_95_LOW": "가상 95% 구간 하한",
    "PLACEBO_95_HIGH": "가상 95% 구간 상한",
    "MC_P_VALUE": "Monte Carlo p값"
})

placebo_summary_kr["분석 모형"] = (
    placebo_summary_kr["분석 모형"]
    .replace(MODEL_LABELS)
)

placebo_summary_kr["플라세보 방식"] = (
    placebo_summary_kr["플라세보 방식"]
    .replace(MODE_LABELS)
)

# KRW_7, U2 98%
view = placebo_summary_kr[
    (placebo_summary_kr["분석 모형"]
     == MODEL_LABELS[MAIN_MODEL]) &
    (placebo_summary_kr["U2 기준"] == "98%")
].copy()

print("■ KRW_7 / U1 87% / U2 98% Placebo 결과")
display(view.reset_index(drop=True).round(5))

# 4개 모형 비교
view = placebo_summary_kr[
    placebo_summary_kr["U2 기준"] == "98%"
].copy()

print("■ U2 98% 기준 4개 모형 비교")
display(view.reset_index(drop=True).round(5))


■ KRW_7 / U1 87% / U2 98% Placebo 결과


,플라세보 방식,분석 모형,U2 기준,실제 위기 조합 수,실제 탐지 건수,실제 탐지율(%),가상 평균 탐지 건수,가상 95% 구간 하한,가상 95% 구간 상한,Monte Carlo p값
0,동일 항로·기간,환율 적용 / 위기 7개,98%,7,5,71.42857,1.73816,0.0,4.0,0.00844
1,전체 KCCI 항로 가중,환율 적용 / 위기 7개,98%,7,5,71.42857,1.53768,0.0,4.0,0.00690


■ U2 98% 기준 4개 모형 비교


,플라세보 방식,분석 모형,U2 기준,실제 위기 조합 수,실제 탐지 건수,실제 탐지율(%),가상 평균 탐지 건수,가상 95% 구간 하한,가상 95% 구간 상한,Monte Carlo p값
0,동일 항로·기간,원본 KCCI / 위기 4개,98%,4,3,75.00000,0.48822,0.0,2.0,0.00236
1,동일 항로·기간,환율 적용 / 위기 4개,98%,4,3,75.00000,0.49682,0.0,2.0,0.00228
2,동일 항로·기간,원본 KCCI / 위기 7개,98%,7,6,85.71429,1.57400,0.0,4.0,0.00024
3,동일 항로·기간,환율 적용 / 위기 7개,98%,7,5,71.42857,1.73816,0.0,4.0,0.00844
4,전체 KCCI 항로 가중,원본 KCCI / 위기 4개,98%,4,3,75.00000,0.71232,0.0,2.0,0.02002
5,전체 KCCI 항로 가중,환율 적용 / 위기 4개,98%,4,3,75.00000,0.81100,0.0,3.0,0.02708
6,전체 KCCI 항로 가중,원본 KCCI / 위기 7개,98%,7,6,85.71429,1.37216,0.0,4.0,0.00028
7,전체 KCCI 항로 가중,환율 적용 / 위기 7개,98%,7,5,71.42857,1.53768,0.0,4.0,0.00690


STEP 15-2. 가상 사건 탐지 건수별 확률분포

In [20]:

# ============================================================
# STEP 15-2. 가상 사건 0~7개 탐지 확률분포
# ============================================================

TARGET_MODEL = "KRW_7"
TARGET_P = "98%"

for mode in [
    "route_matched",
    "all_route_weighted"
]:

    view = placebo_frequency[
        (placebo_frequency["MODE"] == mode) &
        (placebo_frequency["MODEL"] == TARGET_MODEL) &
        (placebo_frequency["U2_PROB"] == TARGET_P)
    ][[
        "DETECTED_EVENTS",
        "SIMULATION_COUNT",
        "PROBABILITY_PCT"
    ]].copy()

    view = view.rename(columns={
        "DETECTED_EVENTS": "탐지 사건 수",
        "SIMULATION_COUNT": "시뮬레이션 횟수",
        "PROBABILITY_PCT": "확률(%)"
    })

    print(f"\n■ {MODE_LABELS[mode]} / KRW_7 / U2 98%")
    display(view.reset_index(drop=True).round(3))



■ 동일 항로·기간 / KRW_7 / U2 98%


,탐지 사건 수,시뮬레이션 횟수,확률(%)
0,0,6236,12.472
1,1,15767,31.534
2,2,16253,32.506
3,3,8780,17.560
4,4,2543,5.086
5,5,403,0.806
6,6,18,0.036
7,7,0,0.000



■ 전체 KCCI 항로 가중 / KRW_7 / U2 98%


,탐지 사건 수,시뮬레이션 횟수,확률(%)
0,0,8748,17.496
1,1,17303,34.606
2,2,14899,29.798
3,3,6792,13.584
4,4,1914,3.828
5,5,314,0.628
6,6,29,0.058
7,7,1,0.002
